In [0]:
!uv pip install -r ../requirements.txt --upgrade

In [0]:
!uv pip install unsloth==2026.7.2 --no-deps

In [0]:
!pip uninstall torchvision -y

In [0]:
%restart_python

In [0]:
%run ../utilities/config

In [0]:
from unsloth import FastLanguageModel
import torch

max_seq_length = non_instruct_max_seq_length

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,          # auto-detect
    load_in_4bit=True,   # QLoRA
)

In [0]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0.0,     # Unsloth is optimized for dropout=0
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [0]:
model.print_trainable_parameters()


In [0]:
with open(non_instruction_dataset) as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")
print(paragraphs[0])

In [0]:
import re

def clean(text):
    text = re.sub(r"\s+", " ", text).strip()
    return text

paragraphs = [clean(p) for p in paragraphs]

# Unsloth/TRL text datasets just need one string per example; a paragraph
# per example is fine at this corpus size (~50 paragraphs).
from datasets import Dataset
raw_dataset = Dataset.from_dict({"text": paragraphs})
raw_dataset

In [0]:
from transformers import AutoTokenizer

total_tokens = 0

for text in raw_dataset["text"]:
    tokens = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]

    total_tokens += len(tokens)

print(f"Total tokens: {total_tokens:,}")


In [0]:
def check_length(example):
    return {
        "length": len(
            tokenizer(
                example["text"],
                add_special_tokens=True,
            )["input_ids"]
        )
    }

length_dataset = raw_dataset.map(check_length)

lengths = length_dataset["length"]

print(f"Max Length : {max(lengths)}")
print(f"Average    : {sum(lengths)/len(lengths):.2f}")
print(f"95th Perc. : {sorted(lengths)[int(0.95*len(lengths))]}")

In [0]:
def filter_long(example):
    return (
        len(
            tokenizer(
                example["text"],
                add_special_tokens=True,
            )["input_ids"]
        )
        <= max_seq_length
    )

dataset = raw_dataset.filter(filter_long)

In [0]:
dataset

In [0]:
from trl import SFTTrainer, SFTConfig
import mlflow
training_args = SFTConfig(
    output_dir="/Volumes/workspace/ai_model/sft_config/stage1_non_instruction",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=raw_dataset,
    args=training_args,
)

#trainer_stats = trainer.train()


In [0]:
import mlflow

# Optional: Set your experiment
mlflow.set_experiment("/Shared/LLM/HRPolicy")

# Enable automatic logging
mlflow.autolog()

# Enable automatic system metrics logging
mlflow.enable_system_metrics_logging()

# Optional: Sample system metrics every 5 seconds
mlflow.set_system_metrics_sampling_interval(5)

with mlflow.start_run(run_name="Stage1_NonInstruction"):

    trainer_stats = trainer.train()

    print(trainer_stats)

In [0]:
import shutil
import tempfile
import os

# Save adapter and tokenizer directly to Volume
model.save_pretrained(stage1_adapter_path)
tokenizer.save_pretrained(stage1_adapter_path)
print(f"✓ Adapter saved to {stage1_adapter_path}")

# Ensure merged directory exists
dbutils.fs.mkdirs(stage1_merged_path)

# Save merged model to temp dir first, then copy to Volume
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Saving merged model to temporary directory: {temp_dir}")
    model.save_pretrained_merged(temp_dir, tokenizer, save_method="merged_16bit")
    
    print(f"Copying merged model to Volume: {stage1_merged_path}")
    # Copy contents from temp dir to Volume
    for item in os.listdir(temp_dir):
        src = os.path.join(temp_dir, item)
        dst = os.path.join(stage1_merged_path, item)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"  Copied: {item}")
    
    print("✓ Merged model saved successfully to {}".format(stage1_merged_path))

In [0]:
print(model.config)
print(model.config._name_or_path)

In [0]:
print(model.config.model_type)

In [0]:
import shutil
import os

cache_dir = "unsloth_compiled_cache"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir, ignore_errors=True)